In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import json
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/valid.spacy
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/__results__.html
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/__notebook__.ipynb
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/__output__.json
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/config.cfg
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/train.spacy
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/custom.css
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-last/tokenizer
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-last/meta.json
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-last/config.cfg
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-last/textcat/model
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-last/textcat/cfg
/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-l

Below code is for claim classification (fact, opinion, suggestion)

In [2]:
'''import spacy
import os

# 1. Define the path to your imported model
# Note: Replace 'sentenceclassifier' with the actual URL slug of your first notebook.
# You can easily find the exact path by clicking through the "Input" folder 
# in the right sidebar and copying the file path for 'model-best'.
MODEL_PATH = "/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-best"

# 2. Check if the path exists (always good for debugging in Kaggle)
if os.path.exists(MODEL_PATH):
    print("✅ Model found! Loading...")
    nlp = spacy.load(MODEL_PATH)
else:
    print(f"❌ Model not found at {MODEL_PATH}. Check your Input folder paths!")

# 3. Run your predictions
def classify_sentence(text):
    doc = nlp(text)
    # Get the label with the highest probability score
    top_label = max(doc.cats, key=doc.cats.get)
    confidence = doc.cats[top_label]
    
    return top_label, confidence

# Test it out
test_sentence = "Ryan is world - renowned scientist"
label, conf = classify_sentence(test_sentence)

print(f"Text: {test_sentence}")
print(f"Classification: {label} (Confidence: {conf:.2f})")'''

'import spacy\nimport os\n\n# 1. Define the path to your imported model\n# Note: Replace \'sentenceclassifier\' with the actual URL slug of your first notebook.\n# You can easily find the exact path by clicking through the "Input" folder \n# in the right sidebar and copying the file path for \'model-best\'.\nMODEL_PATH = "/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-best"\n\n# 2. Check if the path exists (always good for debugging in Kaggle)\nif os.path.exists(MODEL_PATH):\n    print("✅ Model found! Loading...")\n    nlp = spacy.load(MODEL_PATH)\nelse:\n    print(f"❌ Model not found at {MODEL_PATH}. Check your Input folder paths!")\n\n# 3. Run your predictions\ndef classify_sentence(text):\n    doc = nlp(text)\n    # Get the label with the highest probability score\n    top_label = max(doc.cats, key=doc.cats.get)\n    confidence = doc.cats[top_label]\n    \n    return top_label, confidence\n\n# Test it out\ntest_sentence = "Ryan is world - renowned scientist"\n

Below Code is for numeric Hallucination.

In [3]:
'''import spacy

# Load model (make sure you ran: python -m spacy download en_core_web_sm)
nlp = spacy.load("en_core_web_sm")

def check_numeric_drift(ai_claim_text, top_source_sentence):
    NUMERIC_LABELS = {"MONEY", "PERCENT", "DATE", "TIME", "CARDINAL", "QUANTITY"}
    claim_doc = nlp(ai_claim_text)
    claim_numbers = [ent.text for ent in claim_doc.ents if ent.label_ in NUMERIC_LABELS]
    
    if not claim_numbers:
        return "PASS"
        
    for num in claim_numbers:
        if num not in top_source_sentence: 
            return f"DRIFT: '{num}' not found."
    return "PASS"

# --- THE TEST SUITE ---
def run_tests():
    test_cases = [
        {
            "name": "1. Exact Match (Should Pass)",
            "claim": "The CEO bought 500 shares in 2029.",
            "source": "In 2024, the CEO bought 500 shares of the company.",
            "expected": "PASS"
        },
        {
            "name": "2. Blatant Drift (Should Fail)",
            "claim": "The CEO bought 5000 shares in 2024.",
            "source": "In 2024, the CEO bought 500 shares of the company.",
            "expected": "DRIFT: '5000' not found."
        },
        {
            "name": "3. No Numbers at All (Should Pass)",
            "claim": "The CEO bought shares.",
            "source": "The CEO bought shares of the company.",
            "expected": "PASS"
        },
        {
            "name": "4. The Word/Digit Trap (Known Limitation)",
            "claim": "The CEO bought 500 shares.",
            "source": "The CEO bought five hundred shares.",
            "expected": "DRIFT: '500' not found." # Our basic script fails here, which is good to know!
        }
    ]

    print("🏃 Running Numeric Drift Tests...\n")
    passed_tests = 0
    
    for i, test in enumerate(test_cases):
        result = check_numeric_drift(test["claim"], test["source"])
        status = "✅" if result == test["expected"] else "❌"
        
        print(f"{status} Test {i+1}: {test['name']}")
        if result != test["expected"]:
            print(f"   Expected: {test['expected']}")
            print(f"   Got:      {result}")
        else:
            passed_tests += 1
            
    print(f"\n🏁 Results: {passed_tests}/{len(test_cases)} tests passed.")
run_tests()
'''

'import spacy\n\n# Load model (make sure you ran: python -m spacy download en_core_web_sm)\nnlp = spacy.load("en_core_web_sm")\n\ndef check_numeric_drift(ai_claim_text, top_source_sentence):\n    NUMERIC_LABELS = {"MONEY", "PERCENT", "DATE", "TIME", "CARDINAL", "QUANTITY"}\n    claim_doc = nlp(ai_claim_text)\n    claim_numbers = [ent.text for ent in claim_doc.ents if ent.label_ in NUMERIC_LABELS]\n    \n    if not claim_numbers:\n        return "PASS"\n        \n    for num in claim_numbers:\n        if num not in top_source_sentence: \n            return f"DRIFT: \'{num}\' not found."\n    return "PASS"\n\n# --- THE TEST SUITE ---\ndef run_tests():\n    test_cases = [\n        {\n            "name": "1. Exact Match (Should Pass)",\n            "claim": "The CEO bought 500 shares in 2029.",\n            "source": "In 2024, the CEO bought 500 shares of the company.",\n            "expected": "PASS"\n        },\n        {\n            "name": "2. Blatant Drift (Should Fail)",\n          

BELOW IS THE CODE FOR ALIGNMENT MATRIX.

In [4]:
'''import torch
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# 1. Hardware Check: Verify Kaggle has allocated your GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔥 Hardware Engine: {device.upper()}")

# 2. Load Model to GPU
# The 'device' argument ensures embeddings are processed on the graphics card
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
def build_alignment_matrix_kaggle(ai_claims: list, source_sentences: list):
    """
    Computes the semantic alignment matrix using Kaggle's GPU.
    Returns a Pandas DataFrame formatted for the visual frontend.
    """
    
    # 1. Batch encode everything straight to GPU tensors
    # convert_to_tensor=True keeps the data on the GPU for the next step
    claim_embeddings = embedder.encode(ai_claims, convert_to_tensor=True, device=device)
    source_embeddings = embedder.encode(source_sentences, convert_to_tensor=True, device=device)
    
    # 2. Compute the Cosine Similarity Matrix (M x N) in a single GPU operation
    matrix_scores = util.cos_sim(claim_embeddings, source_embeddings)
    
    forensics_data = []
    
    # 3. Extract the S_max and pull the data back to the CPU for the DataFrame
    for i in range(len(ai_claims)):
        # torch.max operates on the GPU tensor
        max_score, max_index = torch.max(matrix_scores[i], dim=0)
        
        forensics_data.append({
            "AI_Claim": ai_claims[i],
            "S_Max_Score": round(max_score.item(), 4), # .item() safely moves the scalar to CPU
            "Source_Index": max_index.item(),
            "Matched_Source_Sentence": source_sentences[max_index.item()]
        })
        
    # 4. Return as a DataFrame for easy Kaggle viewing/exporting
    return pd.DataFrame(forensics_data)

# --- Mock Data ---
mock_claims = [
    "The company was founded in 2020.", 
    "They make a cloud CRM.", 
    "The CEO is Jane Doe."
]

mock_source = [
    "Acme Corp was started in 2020.", 
    "Their main product is a cloud-based CRM designed for sales teams.", 
    "John Smith is the current CEO."
]

# --- Run the Pipeline ---
df_results = build_alignment_matrix_kaggle(mock_claims, mock_source)

# Display the styled DataFrame in the Kaggle Notebook
display(df_results)

# Optional: Export to JSON for your frontend dashboard
# df_results.to_json("hallucination_matrix.json", orient="records")'''

'import torch\nimport pandas as pd\nfrom sentence_transformers import SentenceTransformer, util\n\n# 1. Hardware Check: Verify Kaggle has allocated your GPU\ndevice = \'cuda\' if torch.cuda.is_available() else \'cpu\'\nprint(f"🔥 Hardware Engine: {device.upper()}")\n\n# 2. Load Model to GPU\n# The \'device\' argument ensures embeddings are processed on the graphics card\nembedder = SentenceTransformer(\'all-MiniLM-L6-v2\', device=device)\ndef build_alignment_matrix_kaggle(ai_claims: list, source_sentences: list):\n    """\n    Computes the semantic alignment matrix using Kaggle\'s GPU.\n    Returns a Pandas DataFrame formatted for the visual frontend.\n    """\n    \n    # 1. Batch encode everything straight to GPU tensors\n    # convert_to_tensor=True keeps the data on the GPU for the next step\n    claim_embeddings = embedder.encode(ai_claims, convert_to_tensor=True, device=device)\n    source_embeddings = embedder.encode(source_sentences, convert_to_tensor=True, device=device)\n    \

BELOW IS THE CODE FOR THE TRANSFORMERS-> NLI

In [5]:
'''import pandas as pd
from transformers import pipeline

print("⏳ Loading DeBERTa-v3-small model to GPU... (This takes a few seconds on first run)")

# 1. Load the NLI model 
# device=0 maps to the first Kaggle GPU. If using CPU, change this to device=-1
nli_judge = pipeline(
    "text-classification", 
    model="cross-encoder/nli-deberta-v3-small", 
    device=0 
)

# 2. Define the logic function
def get_nli_verdict(source_sentence, ai_claim):
    """
    Pairs the text and asks DeBERTa to judge the logic.
    Returns: 'Entailed', 'Contradicted', or 'Neutral'
    """
    result = nli_judge({"text": source_sentence, "text_pair": ai_claim})
    
    # Update the map to catch the actual strings returned by the pipeline
    label_map = {
        "contradiction": "Contradicted", 
        "entailment": "Entailed", 
        "neutral": "Neutral",
        # Keep the old ones just in case Kaggle caches an older config version
        "LABEL_0": "Contradicted", 
        "LABEL_1": "Entailed", 
        "LABEL_2": "Neutral"
    }
    
    # Use .lower() to make it case-insensitive, just to be completely safe
    best_label = result['label'].lower() if result['label'] not in label_map else result['label']
    
    # If the lowercased version is in our map, use it
    if best_label in label_map:
        mapped_verdict = label_map[best_label]
    else:
        # Fallback if the model outputs something completely unexpected
        mapped_verdict = result['label'] 
        
    confidence = round(result['score'], 4)
    
    return mapped_verdict, confidence

# 3. Define the test suite
def run_nli_test():
    # The AI claims we want to check
    mock_claims = [
        "The company was founded in 2020.", 
        "They make a cloud CRM.", 
        "The CEO is Jane Doe.",
        "The company has 500 employees." # Added a "Fabrication" test case
    ]

    # The sentences the Semantic Matrix found as the "Alibi"
    mock_source = [
        "Acme Corp was started in 2020.", 
        "Their main product is a cloud-based CRM designed for sales teams.", 
        "John Smith is the current CEO.",
        "Acme Corp was started in 2020." # The source says nothing about employees
    ]

    test_results = []

    print("🔍 Running Inference Engine...")
    
    # zip() lets us iterate through both lists at the same time
    for source, claim in zip(mock_source, mock_claims):
        verdict, confidence = get_nli_verdict(source, claim)
        
        test_results.append({
            "Source_Premise": source,
            "AI_Hypothesis": claim,
            "NLI_Verdict": verdict,
            "Confidence": confidence
        })

    # Wrap the results in a DataFrame for a beautiful Kaggle output
    df_nli = pd.DataFrame(test_results)
    return df_nli

# 4. Execute and display
df_results = run_nli_test()
display(df_results)'''

'import pandas as pd\nfrom transformers import pipeline\n\nprint("⏳ Loading DeBERTa-v3-small model to GPU... (This takes a few seconds on first run)")\n\n# 1. Load the NLI model \n# device=0 maps to the first Kaggle GPU. If using CPU, change this to device=-1\nnli_judge = pipeline(\n    "text-classification", \n    model="cross-encoder/nli-deberta-v3-small", \n    device=0 \n)\n\n# 2. Define the logic function\ndef get_nli_verdict(source_sentence, ai_claim):\n    """\n    Pairs the text and asks DeBERTa to judge the logic.\n    Returns: \'Entailed\', \'Contradicted\', or \'Neutral\'\n    """\n    result = nli_judge({"text": source_sentence, "text_pair": ai_claim})\n    \n    # Update the map to catch the actual strings returned by the pipeline\n    label_map = {\n        "contradiction": "Contradicted", \n        "entailment": "Entailed", \n        "neutral": "Neutral",\n        # Keep the old ones just in case Kaggle caches an older config version\n        "LABEL_0": "Contradicted", \

LETS TRY TO BUILD THE FULL ARCHITECTURE.

In [6]:
import os
import json
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch

print("🚀 Initializing Hallucination Hunter Engine...")

# --- 1. HARDWARE & MODEL INITIALIZATION ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# A. Standard spaCy model for Numeric Drift (NER)
print("Loading standard NER model...")
nlp_ner = spacy.load("en_core_web_sm")

# B. Custom spaCy model for Intent Routing (TextCat)
MODEL_PATH = "/kaggle/input/notebooks/anirbandasbit/sentenceclassifier/output/model-best"
if os.path.exists(MODEL_PATH):
    print("✅ Custom Intent Classifier found! Loading...")
    nlp_intent = spacy.load(MODEL_PATH)
else:
    print(f"❌ ERROR: Model not found at {MODEL_PATH}. Check your Kaggle Input paths!")
    # Fallback to standard model just so the code doesn't crash during testing
    nlp_intent = nlp_ner 

# C. Sentence Transformers & DeBERTa
print("Loading Embedding and NLI models to GPU...")
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
nli_judge = pipeline("text-classification", model="cross-encoder/nli-deberta-v3-small", device=0 if device=='cuda' else -1)

# --- 2. THE PIPELINE FUNCTIONS ---
def classify_intent(text):
    """Uses the custom model to categorize FACT, OPINION, or SUGGESTION"""
    if not os.path.exists(MODEL_PATH): return "FACT" # Safety fallback
    
    doc = nlp_intent(text)
    # Get the label with the highest probability score
    top_label = max(doc.cats, key=doc.cats.get)
    return top_label

def get_nli_verdict(source_sentence, ai_claim):
    result = nli_judge({"text": source_sentence, "text_pair": ai_claim})
    label_map = {"contradiction": "Contradicted", "entailment": "Entailed", "neutral": "Neutral", "LABEL_0": "Contradicted", "LABEL_1": "Entailed", "LABEL_2": "Neutral"}
    best_label = result['label'].lower() if result['label'] not in label_map else result['label']
    return label_map.get(best_label, result['label']), round(result['score'], 4)

def check_numeric_drift(ai_claim_text, top_source_sentence):
    # CRITICAL: Use nlp_ner here, not nlp_intent!
    NUMERIC_LABELS = {"MONEY", "PERCENT", "DATE", "TIME", "CARDINAL", "QUANTITY"}
    claim_numbers = [ent.text for ent in nlp_ner(ai_claim_text).ents if ent.label_ in NUMERIC_LABELS]
    
    if not claim_numbers: return "PASS"
    for num in claim_numbers:
        if num not in top_source_sentence: return f"Drift: '{num}' not found."
    return "PASS"

def build_alignment_matrix(ai_claims, source_sentences):
    claim_embeddings = embedder.encode(ai_claims, convert_to_tensor=True, device=device)
    source_embeddings = embedder.encode(source_sentences, convert_to_tensor=True, device=device)
    matrix_scores = util.cos_sim(claim_embeddings, source_embeddings)
    
    forensics = []
    for i in range(len(ai_claims)):
        max_score, max_index = torch.max(matrix_scores[i], dim=0)
        forensics.append({
            "S_Max": round(max_score.item(), 4),
            "Source_Index": max_index.item(),
            "Matched_Sentence": source_sentences[max_index.item()]
        })
    return forensics

# --- 3. THE MASTER ORCHESTRATOR ---
def evaluate_response(ai_claims, source_sentences):
    print("🔍 Analyzing response...")
    matrix_data = build_alignment_matrix(ai_claims, source_sentences)
    results = []
    
    for i, claim in enumerate(ai_claims):
        # 1. Dynamic Intent Classification using your custom model
        intent = classify_intent(claim)
        
        # 2. Get Alibi from the Matrix
        alibi = matrix_data[i]["Matched_Sentence"]
        s_max = matrix_data[i]["S_Max"]
        
        evaluation = {"claim": claim, "intent": intent, "taxonomy": "TBD"}
        
        # 3. Route based on the custom model's output
        # (Assuming your model outputs "FACT", "OPINION", or "SUGGESTION")
        if intent.upper() == "FACT":
            drift = check_numeric_drift(claim, alibi)
            if drift != "PASS":
                evaluation.update({"taxonomy": "Numeric Drift", "error": drift, "alibi": alibi})
            else:
                nli, conf = get_nli_verdict(alibi, claim)
                if nli == "Entailed": evaluation["taxonomy"] = "Verified Fact"
                elif nli == "Contradicted": evaluation["taxonomy"] = "Contradiction"
                elif nli == "Neutral": evaluation["taxonomy"] = "Extrapolation" if s_max > 0.6 else "Fabrication"
                evaluation.update({"nli": nli, "confidence": conf, "similarity": s_max, "alibi": alibi})
                
        elif intent.upper() == "OPINION":
            if s_max > 0.7:
                taxonomy = "Grounded Opinion"
            elif s_max > 0.4:
                taxonomy = "Weakly Grounded Opinion"
            else:
                taxonomy = "Ungrounded Opinion"
        
            evaluation.update({
                "taxonomy": taxonomy,
                "similarity": s_max,
                "alibi": alibi
            })
        
        
        else:
            # SUGGESTION / unknown
            if s_max > 0.6:
                taxonomy = "Relevant Suggestion"
            else:
                taxonomy = "Irrelevant / Hallucinated Suggestion"
        
            evaluation.update({
                "taxonomy": taxonomy,
                "similarity": s_max,
                "alibi": alibi
            })

        results.append(evaluation)
        
    return json.dumps(results, indent=2)

# --- 4. EXECUTE THE END-TO-END TEST ---
# The backend is now completely autonomous. It reads the claims,
# figures out the intents on its own, finds the alibis, and judges them.

# Taking the sentences and AI-response.
# source = [
#     "The Secret Life of Bees has genre Teen drama.",
#     "Teen drama has examples: A Walk to RememberA Walk to Remember is produced by Hunt Lowry Hunt Lowry produced A Walk to Remember.",
#     "A Walk to Remember is a/an Film"
# ]

# claims = [
#     "I think the Secret Life of Bees is a fantastic book,",
#     "and you should also read A Walk to remember."
# ]
def intake(know, resp):
    source = extract_claims(know)
    claims = extract_claims(resp)

    final_json = evaluate_response(claims, source)
    print("\n🎯 FINAL OUTPUT PAYLOAD:\n")
    print(final_json)

🚀 Initializing Hallucination Hunter Engine...
Loading standard NER model...
✅ Custom Intent Classifier found! Loading...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_pipeline' (0.0.0) was trained with spaCy v3.8.14 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Loading Embedding and NLI models to GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

final_report = hunter.run_pipeline(
    question="Could you recommend any books like The Secret Life of Bees?", 
    source_knowledge="The Secret Life of Bees has genre Teen drama. Teen drama has examples: A Walk to RememberA Walk to Remember is produced by Hunt LowryHunt Lowry produced A Walk to Remember. A Walk to Remember is a/an Film", 
    ai_response="I think the Secret Life of Bees is a fantastic book,and you should also read A Walk to remember."
)


In [7]:
import spacy

nlp = spacy.load("en_core_web_sm")

def has_verb(tokens):
    return any(tok.pos_ in {"VERB", "AUX"} for tok in tokens)

def extract_claims(text):
    split_words = {"and", "but", "also", "however"}
    doc = nlp(text)
    claims = []

    for sent in doc.sents:
        current_chunk = []  # store TOKENS
        tokens = list(sent)

        i = 0
        while i < len(tokens):
            token = tokens[i]

            if token.text.lower() in split_words and token.dep_ == "cc":
                left = current_chunk
                right = tokens[i+1:]

                # ✅ split only if both sides have verbs
                if has_verb(left) and has_verb(right):
                    if current_chunk:
                        claims.append(" ".join(tok.text for tok in current_chunk).strip())
                        current_chunk = []
                    i += 1
                    continue

            current_chunk.append(token)
            i += 1

        if current_chunk:
            claims.append(" ".join(tok.text for tok in current_chunk).strip())

    # clean small/noisy claims
    claims = [c for c in claims if len(c) > 5]

    return claims




In [8]:
know = "John Lennon wrote Yellow Submarine. Yellow Submarine is written by Paul McCartney"
resp = "I like all of Let It Be."
intake(know, resp)

🔍 Analyzing response...

🎯 FINAL OUTPUT PAYLOAD:

[
  {
    "claim": "I like all of Let It Be .",
    "intent": "SUGGESTION",
    "taxonomy": "Irrelevant / Hallucinated Suggestion",
    "similarity": 0.142,
    "alibi": "Yellow Submarine is written by Paul McCartney"
  }
]


In [9]:
test_sentences = [

    "Iron Man is starring Robert Downey Jr.Robert Downey Jr. starred in Zodiac (Crime Fiction Film)Zodiac (Crime Fiction Film) is starring Jake Gyllenhaal",
    "I'm not a fan of crime movies, but I did know that RDJ starred in Zodiac with Tom Hanks."
]

# TEST
for text in test_sentences:
    print(text)
    claims = extract_claims(text)
    print(type(claims))
    for i, c in enumerate(claims, 1):
        print(f"[Claim {i}] → {c}")

Iron Man is starring Robert Downey Jr.Robert Downey Jr. starred in Zodiac (Crime Fiction Film)Zodiac (Crime Fiction Film) is starring Jake Gyllenhaal
<class 'list'>
[Claim 1] → Iron Man is starring Robert Downey Jr. Robert Downey Jr. starred in Zodiac ( Crime Fiction Film)Zodiac
[Claim 2] → ( Crime Fiction Film ) is starring Jake Gyllenhaal
I'm not a fan of crime movies, but I did know that RDJ starred in Zodiac with Tom Hanks.
<class 'list'>
[Claim 1] → I 'm not a fan of crime movies ,
[Claim 2] → I did know that RDJ starred in Zodiac with Tom Hanks .
